<a href="https://colab.research.google.com/github/2303a51899/RL/blob/main/RL_LAB_09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
gamma = 0.9

pi_S0_down = 0.9
pi_S0_right = 0.1
pi_S1_right = 0.9
pi_S1_up = 0.1

R_S0_down = -1.0
R_S0_right = -1.0
R_S1_right = 100.0
R_S1_up = -1.0

const_S0 = pi_S0_down * R_S0_down + pi_S0_right * R_S0_right
coeff_V1 = pi_S0_down * gamma

const_S1 = pi_S1_right * R_S1_right + pi_S1_up * R_S1_up
coeff_V0 = pi_S1_up * gamma

V_S1 = (const_S1 + coeff_V0 * const_S0) / (1 - coeff_V0 * coeff_V1)
V_S0 = const_S0 + coeff_V1 * V_S1

Q_S0_down = R_S0_down + gamma * V_S1
Q_S0_right = R_S0_right + gamma * 0.0
Q_S1_right = R_S1_right + gamma * 0.0
Q_S1_up = R_S1_up + gamma * V_S0

V_S0_verified = pi_S0_down * Q_S0_down + pi_S0_right * Q_S0_right
V_S1_verified = pi_S1_right * Q_S1_right + pi_S1_up * Q_S1_up

print("ALGEBRAIC SOLUTION")
print(f"V(S0)         : {V_S0:.2f} (Exact: {V_S0:.6f})")
print(f"V(S1)         : {V_S1:.2f} (Exact: {V_S1:.6f})")
print("-" * 40)
print(f"Q(S0, Down)   : {Q_S0_down:.2f}")
print(f"Q(S0, Right)  : {Q_S0_right:.2f}")
print(f"Q(S1, Right)  : {Q_S1_right:.2f}")
print(f"Q(S1, Up)     : {Q_S1_up:.2f}")
print("-" * 40)
print(f"Verified V(S0): {V_S0_verified:.2f}")
print(f"Verified V(S1): {V_S1_verified:.2f}")

ALGEBRAIC SOLUTION
V(S0)         : 77.47 (Exact: 77.466293)
V(S1)         : 96.87 (Exact: 96.871966)
----------------------------------------
Q(S0, Down)   : 86.18
Q(S0, Right)  : -1.00
Q(S1, Right)  : 100.00
Q(S1, Up)     : 68.72
----------------------------------------
Verified V(S0): 77.47
Verified V(S1): 96.87


In [4]:
gamma = 0.9
theta = 0.000001

transitions = {
    'S0': {
        'Down': [(0.1, 'Wall', -1.0), (0.9, 'S1', -1.0)],
        'Right': [(1.0, 'Wall', -1.0)],
    },
    'S1': {
        'Right': [(0.9, 'G', 100.0), (0.1, 'S0', -1.0)],
        'Up': [(1.0, 'S0', -1.0)],
    },
}

policy = {'S0': 'Down', 'S1': 'Right'}

V = {'S0': 0.0, 'S1': 0.0, 'Wall': 0.0, 'G': 0.0}

iterations = 0
while True:
  iterations += 1
  delta = 0.0
  V_new = V.copy()

  for s in ['S0', 'S1']:
    action = policy[s]
    expected_value = 0.0
    for prob, next_s, reward in transitions[s][action]:
      expected_value += prob * (reward + gamma * V[next_s])

    V_new[s] = expected_value
    delta = max(delta, abs(V_new[s] - V[s]))

  V = V_new
  if delta < theta:
    break


def compute_q(state, action):
  q_val = 0.0
  for prob, next_s, reward in transitions[state][action]:
    q_val += prob * (reward + gamma * V[next_s])
  return q_val


Q_S0_down = compute_q('S0', 'Down')
Q_S0_right = compute_q('S0', 'Right')
Q_S1_right = compute_q('S1', 'Right')
Q_S1_up = compute_q('S1', 'Up')

print("ITERATIVE POLICY EVALUATION ")
print(f"Number of evaluation iterations : {iterations}")
print(f"V(S0)                           : {V['S0']:.6f} ({V['S0']:.2f})")
print(f"V(S1)                           : {V['S1']:.6f} ({V['S1']:.2f})")
print("-" * 50)
print(f"Q(S0, Down)                     : {Q_S0_down:.6f} ({Q_S0_down:.2f})")
print(f"Q(S0, Right)                    : {Q_S0_right:.6f} ({Q_S0_right:.2f})")
print(f"Q(S1, Right)                    : {Q_S1_right:.6f} ({Q_S1_right:.2f})")
print(f"Q(S1, Up)                       : {Q_S1_up:.6f} ({Q_S1_up:.2f})")
print("-" * 50)
print("Policy-Consistency Checks:")
print(f"V(S0) == Q(S0, Down)  : {abs(V['S0'] - Q_S0_down) < 1e-9}")
print(f"V(S1) == Q(S1, Right) : {abs(V['S1'] - Q_S1_right) < 1e-9}")

ITERATIVE POLICY EVALUATION 
Number of evaluation iterations : 15
V(S0)                           : 77.466292 (77.47)
V(S1)                           : 96.871966 (96.87)
--------------------------------------------------
Q(S0, Down)                     : 77.466293 (77.47)
Q(S0, Right)                    : -1.000000 (-1.00)
Q(S1, Right)                    : 96.871966 (96.87)
Q(S1, Up)                       : 68.719663 (68.72)
--------------------------------------------------
Policy-Consistency Checks:
V(S0) == Q(S0, Down)  : False
V(S1) == Q(S1, Right) : True
